# Exercise 1.5 — The Depolarizing Channel

**Chapter 1: Mathematical Foundations** &nbsp;|&nbsp; *Section 1.3: Quantum Channels and Complete Positivity*

---

## Motivation

The **depolarizing channel** is the simplest model of symmetric noise: it replaces a quantum state with the maximally mixed state with some probability $p$.  It arises naturally from the **Pauli twirl** — averaging any single-qubit channel over the Pauli group yields a depolarizing channel.  This universality makes it central to error analysis in quantum computing, randomized benchmarking, and the theory of quantum error correction thresholds.

## Exercise Statement

The single-qubit depolarizing channel is $\mathcal{E}_p(\rho) = (1-p)\rho + (p/2)I$, with equivalent Kraus form:

$$
\mathcal{E}_p(\rho) = \left(1-\frac{3p}{4}\right)\rho + \frac{p}{4}\bigl(X\rho X + Y\rho Y + Z\rho Z\bigr).
$$

**(a)** Show that $\mathcal{E}_p$ is **unital**: $\mathcal{E}_p(I/2) = I/2$.

**(b)** Verify the Kraus completeness relation $\sum_k K_k^\dagger K_k = I$.

**(c)** Compute the entanglement fidelity $F_e = \langle\Phi^+|(\mathcal{E}_p \otimes \mathrm{id})|\Phi^+\rangle$ and show $F_e = 1 - 3p/4$.

## Solution

### Part (a): Unitality

A channel is **unital** if it preserves the identity: $\mathcal{E}(I) = I$.  For the Pauli-Kraus form, we need to evaluate $\sigma_j \cdot (I/2) \cdot \sigma_j$ for each Pauli $\sigma_j \in \{X, Y, Z\}$.

Since each Pauli matrix is unitary ($\sigma_j^\dagger \sigma_j = I$), we have:

$$
\sigma_j \frac{I}{2} \sigma_j = \frac{1}{2}\sigma_j \sigma_j = \frac{I}{2}.
$$

Substituting into the channel:

$$
\mathcal{E}_p\!\left(\frac{I}{2}\right) = \left(1-\frac{3p}{4}\right)\frac{I}{2} + \frac{p}{4}\cdot 3 \cdot \frac{I}{2} = \frac{I}{2}\left(1-\frac{3p}{4}+\frac{3p}{4}\right) = \frac{I}{2}. \quad \checkmark
$$

**Interpretation:** Unitality means the channel preserves the maximally mixed state — it does not "cool" the system (unlike amplitude damping).  The depolarizing channel adds noise symmetrically in all directions.

### Part (b): Kraus completeness

The Kraus operators are $K_0 = \sqrt{1-3p/4}\,I$ and $K_j = \sqrt{p/4}\,\sigma_j$ for $j \in \{1,2,3\}$.

$$
\sum_k K_k^\dagger K_k = \left(1-\frac{3p}{4}\right)I + \frac{p}{4}\sum_{j=1}^3 \sigma_j^2 = \left(1-\frac{3p}{4}\right)I + \frac{3p}{4}I = I. \quad \checkmark
$$

### Part (c): Entanglement fidelity

The entanglement fidelity measures how well the channel preserves entanglement with a reference system.  For a maximally entangled state $|\Phi^+\rangle = (|00\rangle + |11\rangle)/\sqrt{2}$:

$$
F_e = \sum_k |\langle\Phi^+|(K_k \otimes I)|\Phi^+\rangle|^2.
$$

Using the identity $\langle\Phi^+|(A \otimes I)|\Phi^+\rangle = \mathrm{Tr}(A)/2$:

- $K_0 = \sqrt{1-3p/4}\,I$: $|\mathrm{Tr}(K_0)/2|^2 = (1-3p/4)$.
- $K_j = \sqrt{p/4}\,\sigma_j$: $|\mathrm{Tr}(\sigma_j)/2|^2 = 0$ (Paulis are traceless).

$$
\boxed{F_e = 1 - \frac{3p}{4}.}
$$

At $p = 0$ (no noise): $F_e = 1$.  At $p = 1$ (full depolarization): $F_e = 1/4$ — the minimum for a single-qubit channel, corresponding to the completely depolarizing channel.

---
## Symbolic Verification (SymPy)

In [ ]:
import sympy as sp

p = sp.Symbol('p', real=True, positive=True)
sx = sp.Matrix([[0,1],[1,0]])
sy = sp.Matrix([[0,-sp.I],[sp.I,0]])
sz = sp.Matrix([[1,0],[0,-1]])
I2 = sp.eye(2)

# Part (a): Unitality
rho = I2 / 2
result = (1 - sp.Rational(3,4)*p)*rho
for s in [sx, sy, sz]:
    result += (p/4) * s * rho * s
assert sp.simplify(result - rho) == sp.zeros(2)
print("Part (a): E_p(I/2) = I/2 ✓")

# Part (b): Completeness
completeness = (1 - sp.Rational(3,4)*p)*I2
for s in [sx, sy, sz]:
    completeness += (p/4) * s * s
assert sp.simplify(completeness - I2) == sp.zeros(2)
print("Part (b): Σ K†K = I ✓")

# Part (c): Fe = 1 - 3p/4
Fe = 1 - sp.Rational(3,4)*p
print(f"Part (c): F_e = {Fe}")

---
## Numerical Verification

In [ ]:
import numpy as np

sx = np.array([[0,1],[1,0]], dtype=complex)
sy = np.array([[0,-1j],[1j,0]])
sz = np.array([[1,0],[0,-1]], dtype=complex)
I2 = np.eye(2, dtype=complex)
paulis = [sx, sy, sz]

phi_plus = np.array([1,0,0,1], dtype=complex) / np.sqrt(2)

for p_val in [0.0, 0.1, 0.25, 0.5, 0.75, 1.0]:
    # Apply channel to half of |Φ+⟩
    rho_out = np.zeros((4,4), dtype=complex)
    rho_in = np.outer(phi_plus, phi_plus.conj())
    
    # K0 ⊗ I
    K0 = np.sqrt(1 - 3*p_val/4) * I2
    K0_full = np.kron(K0, I2)
    rho_out += K0_full @ rho_in @ K0_full.conj().T
    
    for s in paulis:
        Kj = np.sqrt(p_val/4) * s
        Kj_full = np.kron(Kj, I2)
        rho_out += Kj_full @ rho_in @ Kj_full.conj().T
    
    Fe = (phi_plus.conj() @ rho_out @ phi_plus).real
    Fe_pred = 1 - 3*p_val/4
    
    print(f"p={p_val:.2f}: F_e = {Fe:.4f} (pred {Fe_pred:.4f})")
    assert np.isclose(Fe, Fe_pred)

print("\nAll assertions passed. ✓")